# Decoding *Wolffia globosa* Biomass Yield from Phytomicrobiome Signatures
### Using Interpretable Machine Learning Models

**Anubrajo Ghosh & Prabrisha Basu**  
Postgraduate and Research Department of Microbiology, St. Xavier's College (Autonomous), Kolkata

---
This notebook implements the CLR &rarr; XGBoost &rarr; TreeSHAP pipeline described in the abstract, on a literature-grounded **synthetic** phytomicrobiome cohort (n = 450 tank samples). Each processing step is isolated in its own cell so the pipeline can be followed, re-run, and audited independently.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_curve, auc, confusion_matrix,
                              recall_score, precision_score)
import shap

RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

## 2. Define the simulated phytomicrobiome taxa
Each taxon is assigned a `(mean_abundance_when_high_competence, mean_abundance_when_low, std)` profile. *Ensifer adhaerens* — the B12-provisioning keystone symbiont central to the Wolffia BioPulse rationale — is deliberately given the strongest, tightest separation between classes, so it is expected to emerge as the top predictive 'sentinel' taxon.

In [2]:
# Plant-growth-promoting bacteria (PGPB): N-fixation, P-solubilization,
# IAA production, siderophores, cobalamin provisioning
beneficial_profiles = {
    'Ensifer_adhaerens':       (0.075, 0.012, 0.018),
    'Bacillus_megaterium':     (0.045, 0.020, 0.028),
    'Pseudomonas_fluorescens': (0.040, 0.022, 0.028),
    'Azospirillum_brasilense': (0.040, 0.020, 0.030),
    'Rhizobium_sp':            (0.038, 0.020, 0.030),
}

# Deleterious / opportunistic taxa: frond decay, biofilm fouling,
# stagnant-water culture collapse
pathogenic_profiles = {
    'Aeromonas_hydrophila':     (0.012, 0.040, 0.025),
    'Pseudomonas_aeruginosa':   (0.010, 0.042, 0.025),
    'Flavobacterium_columnare': (0.012, 0.035, 0.025),
    'Chryseobacterium_sp':      (0.012, 0.032, 0.025),
    'Elizabethkingia_sp':       (0.012, 0.030, 0.025),
}

# Commensal / functionally neutral background taxa
neutral_taxa = ['Sphingomonas_sp', 'Microbacterium_sp']

TAXA_COLS = list(beneficial_profiles) + list(pathogenic_profiles) + neutral_taxa
print(f'{len(TAXA_COLS)} taxa defined.')

12 taxa defined.


## 3. Dataset generator function
`Biomass_Status = 1` denotes 'High Doubling Competence' (doubling time ≲ 36–42 h, healthy frond density); `0` denotes 'Low Doubling Competence' (stagnant/declining culture, chlorosis onset). A 15% outlier rate injects biological/technical noise so the classes are not trivially separable.

In [3]:
def generate_phytomicrobiome_data(n_samples=450):
    """Simulates a frond-associated phytomicrobiome + cultivation dataset for
    Wolffia globosa grown in replicate aquaculture tanks. Taxa are reported as
    relative (compositional) abundances, mirroring real amplicon-sequencing output.
    """
    np.random.seed(RANDOM_STATE)
    data = []

    for i in range(n_samples):
        status = 1 if i < n_samples // 2 else 0  # balanced classes
        is_outlier = np.random.random() < 0.15
        eff = 1 - status if is_outlier else status

        row = {
            'SampleID': f'WGL_{str(i+1).zfill(3)}',
            'Water_Temp_C': round(np.random.normal(27, 3), 1),
            'pH': round(np.random.normal(6.9, 0.5) if eff == 1 else np.random.normal(7.3, 0.55), 2),
            'Ammonium_N_mgL': round(max(0.1, np.random.normal(9, 3) if eff == 1 else np.random.normal(13, 5)), 2),
            'Biomass_Status': status
        }

        for sp, (avg_high, avg_low, std) in beneficial_profiles.items():
            avg = avg_high if eff == 1 else avg_low
            row[sp] = max(0.0001, np.random.normal(avg, std))

        for sp, (avg_high, avg_low, std) in pathogenic_profiles.items():
            avg = avg_high if eff == 1 else avg_low
            row[sp] = max(0.0001, np.random.normal(avg, std))

        for sp in neutral_taxa:
            row[sp] = max(0.001, np.random.normal(0.04, 0.03))

        # Shannon-style Diversity Index: richer, more balanced communities
        # track healthier/faster-doubling cultures in this synthetic cohort
        row['Diversity_Index'] = round(
            np.random.normal(2.6, 0.4) if eff == 1 else np.random.normal(1.7, 0.5), 2
        )
        data.append(row)

    df = pd.DataFrame(data)
    return df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)  # shuffle

print('Generator function defined.')

Generator function defined.


## 4. Generate the cohort and save to CSV

In [4]:
df = generate_phytomicrobiome_data(450)
df.to_csv('wolffia_phytomicrobiome_study.csv', index=False)
print(f'✅ Synthetic Wolffia globosa phytomicrobiome cohort generated: {df.shape[0]} samples, {df.shape[1]} columns.')
df.head(3)

✅ Synthetic Wolffia globosa phytomicrobiome cohort generated: 450 samples, 18 columns.


,SampleID,Water_Temp_C,pH,Ammonium_N_mgL,Biomass_Status,Ensifer_adhaerens,Bacillus_megaterium,Pseudomonas_fluorescens,Azospirillum_brasilense,Rhizobium_sp,Aeromonas_hydrophila,Pseudomonas_aeruginosa,Flavobacterium_columnare,Chryseobacterium_sp,Elizabethkingia_sp,Sphingomonas_sp,Microbacterium_sp,Diversity_Index
0,WGL_433,26.2,8.26,12.54,0,0.023045,0.000100,0.013058,0.036781,0.037543,0.056041,0.035489,0.014860,0.005928,0.022041,0.051712,0.070201,2.31
1,WGL_329,25.1,8.40,17.60,0,0.004975,0.017576,0.010900,0.006963,0.031006,0.035784,0.032885,0.017708,0.071944,0.068889,0.066118,0.001405,1.45
2,WGL_205,24.4,7.62,18.12,1,0.000100,0.003136,0.000100,0.008413,0.000100,0.055324,0.039531,0.032871,0.047339,0.051283,0.030921,0.059639,1.87


## 5. Centered log-ratio (CLR) transform
Relative-abundance taxa live on the Aitchison simplex (they sum to a constant), which induces spurious negative correlations under ordinary Euclidean statistics. CLR projects each sample's taxon vector onto unconstrained Euclidean space by dividing (in log-space, subtracting) by the per-sample geometric mean.

In [5]:
def clr_transform(df, cols, pseudocount=1e-6):
    comp = df[cols].copy() + pseudocount
    log_comp = np.log(comp)
    geo_mean_log = log_comp.mean(axis=1)
    clr = log_comp.sub(geo_mean_log, axis=0)
    clr.columns = [f'CLR_{c}' for c in cols]
    return clr

print('CLR transform function defined.')

CLR transform function defined.


## 6. Build the feature matrix and train/test split

In [6]:
clinical_cols = ['Water_Temp_C', 'pH', 'Ammonium_N_mgL', 'Diversity_Index']
clr_df = clr_transform(df, TAXA_COLS)

X = pd.concat([df[clinical_cols].reset_index(drop=True),
               clr_df.reset_index(drop=True)], axis=1)
y = df['Biomass_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (360, 16), Test: (90, 16)


## 7. Train a regularized XGBoost classifier
L1 (`reg_alpha`) and L2 (`reg_lambda`) penalties handle multi-taxa collinearity introduced by the CLR transform (all CLR features share a common geometric-mean term).

In [7]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    reg_alpha=0.5,   # L1
    reg_lambda=1.0,  # L2
    eval_metric='logloss'
)
model.fit(X_train, y_train)
print('🌱 Model trained.')

🌱 Model trained.


## 8. Apply a tuned decision threshold and compute metrics
A threshold of 0.40 (rather than the default 0.50) trades a little precision for recall, favouring fewer missed low-yield/crash cases.

In [8]:
THRESHOLD = 0.40

y_probs = model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= THRESHOLD).astype(int)

acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)

print(f'📈 Threshold: {THRESHOLD}')
print(f'Accuracy:  {acc:.2%}')
print(f'Recall:    {rec:.2%}')
print(f'Precision: {prec:.2%}')

📈 Threshold: 0.4
Accuracy:  78.89%
Recall:    84.44%
Precision: 76.00%


## 9. Chart 1 — Confusion matrix

In [9]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title(f'Confusion Matrix (Threshold: {THRESHOLD})\nRecall: {rec:.2%}')
plt.ylabel('Actual Biomass Status')
plt.xlabel('Predicted Biomass Status')
plt.savefig('1_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_1160\3437834946.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Chart 2 — Taxonomic & cultivation correlation matrix

In [10]:
numeric_df = df.drop(columns=['SampleID'], errors='ignore').select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    mask=np.triu(np.ones_like(corr_matrix, dtype=bool)),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    annot_kws={'size': 8}, linewidths=.5
)
plt.title('Taxonomic & Cultivation Correlation Matrix', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('2_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_1160\292586921.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Chart 3 — XGBoost gain-based feature importance

In [11]:
importances = pd.Series(model.feature_importances_, index=X.columns)
top12 = importances.sort_values(ascending=False).head(12)[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top12.index, top12.values, color='#2E86AB')
plt.xlabel('Importance Score')
plt.title('Top 12 Predictors of Biomass Doubling Competence (XGBoost)')
plt.tight_layout()
plt.savefig('3_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_1160\376776849.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Compute TreeSHAP values
TreeSHAP derives game-theoretic Shapley attributions per prediction, isolating each taxon's directional contribution to the model output.

In [12]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
print('📊 SHAP values computed for', X_test.shape[0], 'test samples.')

📊 SHAP values computed for 90 test samples.


## 13. Chart 4 — SHAP summary plot

In [13]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary: Impact on Biomass Doubling Competence')
plt.tight_layout()
plt.savefig('4_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_1160\2830455593.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test, show=False)
C:\Temp\ipykernel_1160\2830455593.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Chart 5 — ROC curve

In [14]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}', color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='navy')
idx = np.argmin(np.abs(thresholds_roc - THRESHOLD))
plt.scatter(fpr[idx], tpr[idx], color='red', s=100,
            label=f'Chosen Threshold ({THRESHOLD})', zorder=5)
plt.title('ROC Curve')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('5_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Temp\ipykernel_1160\532037473.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 15. Summary

In [15]:
print('✅ ANALYSIS SUMMARY COMPLETE')
print(f'Threshold: {THRESHOLD}')
print(f'Accuracy:  {acc:.2%}')
print(f'Recall:    {rec:.2%}')
print(f'Precision: {prec:.2%}')
print(f'AUC-ROC:   {roc_auc:.2f}')
print('Files generated: 1_confusion_matrix.png, 2_correlation_heatmap.png, '
      '3_feature_importance.png, 4_shap_summary.png, 5_roc_curve.png')

✅ ANALYSIS SUMMARY COMPLETE
Threshold: 0.4
Accuracy:  78.89%
Recall:    84.44%
Precision: 76.00%
AUC-ROC:   0.91
Files generated: 1_confusion_matrix.png, 2_correlation_heatmap.png, 3_feature_importance.png, 4_shap_summary.png, 5_roc_curve.png
